#  Mini-Project 2 — London Transport Connectivity Analysis

**DS105A Mini-Project 2 – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: 03/12/2025
- 👤 Name: Ahmad Abdulaziz
- 🔢 Candidate Number: 65251
- 🎯 Purpose: To answer the question - **Where are the areas in London with poor transport connectivity?**


</div>

# NB01: Data Collection

⚙️ **Importing libraries**

In [1]:
import os
import json
import requests

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

True

**CONSTANTS**

(Variables that will be used throughout the notebook and won't change.)

In [ ]:
BASE_URL = "https://api.tfl.gov.uk/Journey/JourneyResults/{from}/to/{to}"

API_KEY = os.getenv("API_KEY")

 # Easily found on LSE websites
LSE_POSTCODE = "WC2A 2AE"

# Update if running in the future
CURRENTDATE = "20251201"

## Section 1: Test API Authentication

Most of the data I will use in this project comes from the [Transport for London API](https://api-portal.tfl.gov.uk/). I have already signed up and created an API key from the portal. Then, I copied it to the `.env` file and loaded it with the code above (`load_dotenv()` and `os.getenv()`) so everything should be working fine:

In [3]:
# Test that the API key is loaded without ever looking at it:
if API_KEY:
    print("✅ API key loaded.")
else:
    print("❌ No API key found.")

✅ API key loaded.


Now I will test that the API key is valid by making a request to the API:

In [4]:
# Format the URL with the postcodes:
url = BASE_URL.replace("{from}", LSE_POSTCODE).replace("{to}", LSE_BANKSIDE_HOUSE_HALL)

# Make the request:
response = requests.get(url, headers={"app_key": API_KEY})

# Check if the request was successful:
if response.status_code == 200:
    print("✅ API request successful!")
else:
    print(f"❌ API request failed with status code {response.status_code}")


✅ API request successful!


## Section 2: Collecting the journey duration from each MSOA to Central London

### Section 2.1: Creating my sample of postcodes to use

In [13]:
# Find a random sample of the postcodes for each MSOA

postcodes = pd.read_csv("data/raw/london_postcodes-ons-postcodes-directory-feb22.csv")
postcodes_filtered = postcodes[postcodes["doterm"].isna()] # Using adapted code for the "active" postcodes on Moodle instructions
sample = postcodes_filtered.groupby("msoa11")["pcds"].sample(n=1, random_state=1)
sampled_df = postcodes_filtered.loc[sample.index].drop(["pcd", "pcd2"], axis=1)
sampled_df.head()

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,ru11ind,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp
69453,EC4Y 0DR,198001,NaN,E99999999,E99999999,E09000001,E05009297,E43000191,0,531477,...,A1,2D3,51.511396,-0.106765,E37000051,NaN,E23000034,18089,E56000028,E54000029
172315,RM6 5LJ,198001,NaN,E99999999,E99999999,E09000002,E05000029,E43000192,0,548183,...,A1,4A3,51.583846,0.137391,E37000051,NaN,E23000001,5097,E56000028,E54000029
173941,RM8 1NU,200106,NaN,E99999999,E99999999,E09000002,E05000042,E43000192,0,548557,...,A1,4C2,51.568618,0.142100,E37000051,NaN,E23000001,8544,E56000028,E54000029
165202,RM10 7AB,198001,NaN,E99999999,E99999999,E09000002,E05000030,E43000192,0,550401,...,A1,4A3,51.555513,0.168059,E37000051,NaN,E23000001,5575,E56000028,E54000029
174522,RM8 3PR,198001,NaN,E99999999,E99999999,E09000002,E05000040,E43000192,0,548592,...,A1,4A1,51.559955,0.142171,E37000051,NaN,E23000001,4126,E56000028,E54000029


**Personal Reflection Note:**

For my first bit of data collection, which eventually will lead to my choropleth map of London and analysis of variations in travel times within boroughs, I decided to group by MSOA (Middle layer Super Output Area). This was done mainly for practical reasons, as the postcode directory contained over 300,000 postcodes, so it would be unrealistic to pass all of them to the API given each response takes just over a second, meaning if my maths is right it would take over 83 hours to go through every postcode! I wanted my choropleth map to be as granular as possible, so I looked into maybe using LSOAs, but LSOAs were also unrealistically high in numbers, with around 5000 of them in London. Perhaps if I was running this code on my own machine, I would have used LSOAs, but as I later found out, Nuvolos doesn't let you run the application in the background. You need to constantly interact with the tab running Nuvolos, otherwise the session will time out and the API collection will stop (which happened to me on my first attempt). Therefore, I decided to group by MSOA instead, as there are only 983, which is a realistic number for me to pass to the API (in the end it took 70 minutes to collect my data for the 983 MSOAs). From the appendix in the mini-project 2 brief and [the ONS's website](https://www.ons.gov.uk/methodology/geography/ukgeographies/censusgeographies/census2021geographies), I saw that MSOAs each have a standardized size of around 7500 residents (5000 to 15000), which I think still allows for a good level of granularity while also being manageable with my API calls.

Once I had decided to use MSOAs as my geographical hierarchy, I decided to sample 1 postcode from each MSOA to pass to the Journey Planner API. This was again done for practicality's sake to limit how many API calls I needed, but also I think MSOAs are small enough that having only 1 postcode to represent an MSOA shouldn't have too much of a negative impact on my analysis. To select what postcode I would use for each MSOA, I used the pd.sample tool I [discovered in the Pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html), and used the random_state feature to ensure reproducibility, as it creates a "seed" that will ensure that each time you run the code in a specific cell, the sample you got from the first run stays the same in later runs. Before I took my samples, I also ensured that I filtered out all the postcodes that were labelled as not-in-use (with a doterm date), using the code that was on the mini-project 2 brief under "Critical Data Handling Notes". However, as you will see in Section 2.2, some of the postcodes had become out-of-date after the publication of the postcode directory, which posed an issue for calling the TfL API, and was something I had to solve. [I used Claude](https://claude.ai/share/d430e299-7290-4a1e-91f9-1d4f346011e7) in this to specifically help me create code that would save the entire row for each of the sample postcodes, as I had struggled to do this myself initially.

### Section 2.2: Dealing with out-of-date postcodes

**Personal Reflection Note:**

Despite filtering out the postcodes that had a date of termination before creating my sample_df using the code that Jon suggested on Moodle for the mini-project 2 page (under the section "critical data handling notes"), when I collected my data (more to be seen below), some postcodes failed to provide journey data. The postcodes that failed were:

- SE9 9GR
- W1U 8YY
- SW8 9FJ
- E6 9QD
- RM6 4UL
- CR44 1AD
- SW18 2TP
- BR3 9PG

 I then investigated why this was the case, and after looking up the postcodes online I found [this website](https://www.doogal.co.uk/), which told me that these postcodes were no longer in use, but the date of termination was after February 2022, which is when the [postcode directory](https://data.london.gov.uk/dataset/postcode-directory-for-london-exp5p/) we were instructed to use was updated (for example, some had expired in 2023, others in late 2022 etc.). TfL's Journey Planner API wasn't able to generate journeys for these expired postcodes. I did think of leaving it, as it was only 8 of the 983 MSOAs that were impacted, but after the Week 9 lecture and Jon's emphasis on exploring **why** we may have anomalies or missing data and specifically the guidance on slide 20 of the lecture of trying to fix the issue if it's a data collection problem, which I would argue this was, I decided to fix this. Below is the code which I used, which involved me manually creating a list including the failed postcodes, identifying which MSOAs they were in, and then extracting a new sample postcode for these MSOAs. I did have to repeat this a few times with different "random_state"s before I finally got a new sample of 8 postcodes for these MSOAs that wasn't expired. I checked this by manually inputting the newly extracted postcodes into the [aforementioned website](https://www.doogal.co.uk/uk/) each time. I then updated my sampled_df with these new postcodes for the 8 MSOAs before I saved it as a csv file.

When I was collecting data to find the local authority names in NB02 (more on that later), I also found that [updated postcode directories](https://geoportal.statistics.gov.uk/search?q=PRD_ONSPD%20NOV_2025&sort=Date%20Created%7Ccreated%7Cdesc) do exist on the Office for National Statistics' Website, which I may have used instead if we weren't specified as needing to use the postcode directory listed on Moodle as I think this would resolve the issue in a more "elegant" way than manually checking postcodes. However this directory I found on the ONS's website is for the whole of the UK, so probably would have taken longer to load and process, and would involve having to filter the csv myself to only include London postcodes.

In [14]:
failed_postcodes_list = ["SE9 9GR", "W1U 8YY", "SW8 9FJ", "E6 9QD", "RM6 4UL", "CR44 1AD", "SW18 2TP", "BR3 9PG"]
failed_postcodes = sampled_df[sampled_df["pcds"].isin(failed_postcodes_list)]
failed_postcodes

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,ru11ind,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp
209771,SE9 9GR,202112,NaN,E99999999,E99999999,E09000011,E05000219,E43000201,1,542733,...,A1,3D1,51.449793,0.052711,E37000051,NaN,E23000001,14143,E56000010,E54000030
301901,W1U 8YY,202004,NaN,E99999999,E99999999,E09000019,E05000370,E43000209,1,531073,...,A1,3D3,51.524567,-0.112017,E37000051,NaN,E23000001,11070,E56000027,E54000028
250445,SW8 9FJ,202103,NaN,E99999999,E99999999,E09000022,E05000429,E43000212,1,529940,...,A1,2D2,51.475661,-0.130353,E37000051,NaN,E23000001,15107,E56000010,E54000030
53883,E6 9QD,202202,NaN,E99999999,E99999999,E09000025,E05000482,E43000215,1,542855,...,A1,4B1,51.525566,0.057877,E37000051,NaN,E23000001,8245,E56000028,E54000029
172148,RM6 4UL,201702,NaN,E99999999,E99999999,E09000026,E05011237,E43000216,0,547007,...,A1,4B1,51.571962,0.119889,E37000051,NaN,E23000001,14098,E56000028,E54000029
17466,CR44 1AD,200702,NaN,E99999999,E99999999,E09000029,E05000555,E43000219,1,530031,...,A1,4A1,51.382348,-0.132875,E37000051,NaN,E23000001,15853,E56000021,E54000031
227769,SW18 2TP,202006,NaN,E99999999,E99999999,E09000032,E05000612,E43000222,1,525768,...,A1,2D2,51.448330,-0.191511,E37000051,NaN,E23000001,20364,E56000021,E54000031
5596,BR3 9PG,202011,NaN,E99999999,E99999999,E09000006,E05000120,E43000196,1,535885,...,A1,4A1,51.396362,-0.048125,E37000051,NaN,E23000001,10457,E56000010,E54000030


In [15]:
failed_msoa_list = list(failed_postcodes["msoa11"])
failed_msoa_list

['E02000340',
 'E02000575',
 'E02000623',
 'E02000736',
 'E02000769',
 'E02000850',
 'E02000941',
 'E02006787']

In [16]:
updated_failed_msoa_postcodes = postcodes_filtered[postcodes_filtered["msoa11"].isin(failed_msoa_list)]
updated_samples = updated_failed_msoa_postcodes.groupby("msoa11")["pcds"].sample(n=1, random_state=1)
updated_samples_df = postcodes_filtered.loc[updated_samples.index].drop(["pcd", "pcd2"], axis=1)
updated_samples_df

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,ru11ind,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp
208508,SE9 2PQ,198001,NaN,E99999999,E99999999,E09000011,E05000219,E43000201,1,544424,...,A1,6A1,51.450533,0.077056,E37000051,NaN,E23000001,26343,E56000010,E54000030
286662,W1A 3ZG,199806,NaN,E99999999,E99999999,E09000019,E05000370,E43000209,1,531073,...,A1,3D3,51.524565,-0.112042,E37000051,NaN,E23000001,11070,E56000027,E54000028
249102,SW8 1HW,199007,NaN,E99999999,E99999999,E09000022,E05000429,E43000212,0,530493,...,A1,3D2,51.476720,-0.122371,E37000051,NaN,E23000001,8081,E56000010,E54000030
52354,E6 2NG,198001,NaN,E99999999,E99999999,E09000025,E05000493,E43000215,0,542926,...,A1,4B1,51.534767,0.059289,E37000051,NaN,E23000001,12000,E56000028,E54000029
171989,RM6 4FE,199501,NaN,E99999999,E99999999,E09000026,E05011237,E43000216,0,546711,...,A1,4B1,51.577779,0.115861,E37000051,NaN,E23000001,16873,E56000028,E54000029
11341,CR0 4SE,198001,NaN,E99999999,E99999999,E09000029,E05000555,E43000219,0,530211,...,A1,5A3,51.371388,-0.130735,E37000051,NaN,E23000001,18343,E56000021,E54000031
227555,SW18 2BJ,198001,NaN,E99999999,E99999999,E09000032,E05000627,E43000222,0,526056,...,A1,3D2,51.451249,-0.187276,E37000051,NaN,E23000001,20379,E56000021,E54000031
4737,BR3 4LP,198001,NaN,E99999999,E99999999,E09000006,E05000112,E43000196,0,536481,...,A1,5A2,51.402816,-0.039308,E37000051,NaN,E23000001,27232,E56000010,E54000030


In [17]:
sampled_df = sampled_df[~sampled_df["pcds"].isin(failed_postcodes_list)]
sampled_df = pd.concat([sampled_df, updated_samples_df])
len(sampled_df)

983

**Personal Reflection Note**

I used pd.concat here following inspiration from the Week 7 lecture (slide 21), as all I needed to do was remove the 8 "outdated" postcode rows from my previous dataframe and then tack on the new updated rows of postcodes to the end. This works because, when I write my API call below, the order of which postcodes get called first ultimately doesn't matter (e.g., I'm not calling 1 borough first, then another, then another etc.). The list of durations is saved in the order of the updated sampled_df DataFrame so that won't be an issue with "matching" the duration to its corresponding row.

In [10]:
# Creating my CSV file containing the sampled postcodes DataFrame
sampled_df.to_csv("data/raw/sample_postcodes.csv", index=False)

### Section 2.3: Calling the TfL Journey Planner API

In [ ]:
# Creating a list of postcodes to use to call the API
postcodelist = list(sampled_df["pcds"])

In [64]:
len(postcodelist)

983

In [ ]:
list_of_durations = []
for postcode in postcodelist:
    base_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{postcode}/to/WC2A2AE"

    params = {
        "date": CURRENTDATE,
        "time": "0800",
        "app_id": API_KEY
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        test_data = response.json()
        duration = test_data["journeys"][0]["duration"]
        list_of_durations.append(duration)
    else:
        print(f"Failed for {postcode}: {response.status_code}")
        list_of_durations.append(None)

In [ ]:
# Ensuring my list of durations is the same length as my postcode list (aka my dataframe length)
len(list_of_durations) == len(postcodelist)

True

In [15]:
sampled_df["duration_to_central"] = list_of_durations

In [17]:
sampled_df.head()

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp,duration_to_central
69453,EC4Y 0DR,198001,NaN,E99999999,E99999999,E09000001,E05009297,E43000191,0,531477,...,2D3,51.511396,-0.106765,E37000051,NaN,E23000034,18089,E56000028,E54000029,13
172315,RM6 5LJ,198001,NaN,E99999999,E99999999,E09000002,E05000029,E43000192,0,548183,...,4A3,51.583846,0.137391,E37000051,NaN,E23000001,5097,E56000028,E54000029,72
173941,RM8 1NU,200106,NaN,E99999999,E99999999,E09000002,E05000042,E43000192,0,548557,...,4C2,51.568618,0.142100,E37000051,NaN,E23000001,8544,E56000028,E54000029,75
165202,RM10 7AB,198001,NaN,E99999999,E99999999,E09000002,E05000030,E43000192,0,550401,...,4A3,51.555513,0.168059,E37000051,NaN,E23000001,5575,E56000028,E54000029,72
174522,RM8 3PR,198001,NaN,E99999999,E99999999,E09000002,E05000040,E43000192,0,548592,...,4A1,51.559955,0.142171,E37000051,NaN,E23000001,4126,E56000028,E54000029,72


In [ ]:
sampled_df.to_csv("data/raw/sample_postcodes_durations.csv", index=False)

**Personal Reflection Note:**

This code to collect data from the API was inspired by some of the work we did in the Week 4 practice, and also what I learnt in the Dataquest practice we were set in the earlier weeks of the course, specifically the use of list appending to collect data from the API. This code runs runs through each postcode in my sampled_df, and collects the duration of travel from the postcode to LSE, storing each duration in my list of durations (with provisions for errors where the Journey Planner API fails to provide a journey). The duration collected is for the first journey the TfL Journey Planner API presents the user. I decided to do this both for practicality (if I can shave off a few seconds from each API call, with 983 postcodes that can make a huge difference in how long it takes to collect the data!), but also because I [researched the Journey Planner API](https://www.london.gov.uk/who-we-are/what-london-assembly-does/questions-mayor/find-an-answer/tfl-journey-planner-3) and from what I can tell based on [responses given in the London Assembly by officials from TfL at Mayor's Questions](https://www.london.gov.uk/who-we-are/what-london-assembly-does/questions-mayor/find-an-answer/step-free-journey-planner), the journey planner is designed to provide the fastest journey at a given time. Therefore, if I set a journey to start at 08:00, as I did, the journey that will get you to your destination in as short of a time from 08:00 will be selected, which I think is in the spirit of what I'm trying to simulate here, which is the morning peak commute. There's no use in a later journey that may be a few minutes quicker but departs at (for example) 8:45am, as the hypothetical person will miss their work or 9am lecture in that case! Hence, it is for this reason and the practicality that I simply picked the first journey that the TfL Journey Planner API produced for each postcode.

I decided to use the list appending method rather than the more complex data handling tools we were introduced to in the Week 7 lecture and lab because it sufficied for what I was doing, which was extracting the duration of the first journey from each API call. I did use these tools, especially json_normalize, to explore the nested data initially when I first opened the notebook before I committed to my methodology, just to see what the structure of the nested JSON was and what data the TfL Journey Planner API provides (and crucially where it's stored). I deleted this code however, as it would be redundant in my current NB01, but the learning I got from it was used in my final code as I was able to use it to identify that the duration of the first journey was stored in test_data["journeys"][0]["duration"]. When I was writing the for-loop, I did [use Claude](https://claude.ai/share/1259ebdb-7ae6-4e6c-8b03-6e0ef07ed954) to help with my syntax, and it also gave me the help on adding code to deal with errors to ensure my code was more robust. To test my for-loop, as you can see above, I first ran it on a shortened list of my postcodes, before running it on the full list of postcodes extracted from my sampled_df. After data collection, the list was turned into a dataframe column in my original sampled_df dataframe and then saved to a new CSV. The list of durations is in the exact order of my sample postcodes, so this worked well - like a slightly tweaked version of the "2 lists" method we used for the collection of dates and temperatures from the OpenWeather API in the Week 4 practice. Before I saved it, I ensured the length of durations matched the length of my postcodes, and it did!

For transparency, I tested it twice and the data collection for all 983 postcodes took approximately 70 minutes on both tries.

## Section 3: Collecting More Data on Most and Least Deprived Neighbourhoods Per Borough

In this section, I decided to collect more information specifically on the most and least deprived areas in each borough and their journey times and total walking durations to LSE. As I read in the mini-project 2 brief, the Index of Multiple Deprivation (IMD) was included in the postcode directory, however the IMD is measured based on LSOA rather than postcode. Hence, if I took for example the "top 5 most deprived postcodes" in each borough, chances are all 5 would belong to the same LSOA and therefore from the same area, which isn't really helpful and feels like my data collection would just include unnecessary duplication. I also quite liked the idea of comparing extremes, with specifically the **most** deprived and **least** deprived areas in each borough, to see if there are differences in journey times based on deprivation. Hence, I just chose 1 "most deprived postcode" (de facto the first postcode from the most deprived LSOA) and 1 least deprived postcode (same but for the least deprived LSOA) from each borough, and did analysis based on this. This also came with the benefit of keeping the API call simple and quite short, especially when compared to Section 2!

### Section 3.1: Filtering for the most and least deprived areas

In [80]:
most_deprived = postcodes_filtered.loc[postcodes_filtered.groupby('oslaua')['imd'].idxmin()].drop(["pcd", "pcd2"], axis=1)
most_deprived.to_csv("data/raw/most_deprived_postcodes.csv", index=False)
most_deprived_list = list(most_deprived["pcds"])
len(most_deprived_list)

33

In [81]:
least_deprived = postcodes_filtered.loc[postcodes_filtered.groupby('oslaua')['imd'].idxmax()].drop(["pcd", "pcd2"], axis=1)
least_deprived_list = list(least_deprived["pcds"])
len(least_deprived_list)

33

**Personal Reflection Note:**

First, I tried to do this using the methods we had learnt in class, specifically the groupby methods covered in Week 5 and used throughout ever since, but struggled a little bit to get exactly what I wanted which was a dataframe with 33 rows, each containing information for a postcode in the most deprived LSOA in each borough, and then the same for the least deprived. This is when I decided to [ask Claude](https://claude.ai/share/8b28dcbe-f8c1-48d3-b33e-98eef6776a4b) for help, and it gave me a few options of what code I could potentially use, of which this looked the cleanest so I ended up using it and got what I wanted. Claude did use a pandas tool I hadn't come across yet, idxmax, so I asked it to explain what the tool does, and it informed me that it was a tool that "returns the index label (row label) of the row containing the value", which is then useful when combined with the pandas .loc tool as that will precisely filter for the rows that I want.

The code that follows is largely similar to the data collection I used for my earlier API call, with the added dimension of also checking for the duration of walking time for each journey.

### Section 3.2: Calling API for Most and Least Deprived Areas

In [82]:
# Used this to test my code below - left it in for transparency and to show how testing worked
# I did the same for my section 2.3 code too, I just ended up deleting it
testsource = most_deprived_list[:3]
testsource

['E1 7AA', 'IG11 0AG', 'NW11 9EH']

In [ ]:
# Most Deprived First
list_of_durations = []
list_of_walking_time = []
for postcode in most_deprived_list:
    base_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{postcode}/to/WC2A2AE"

    params = {
        "date": CURRENTDATE,
        "time": "0800",
        "app_id": API_KEY
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        test_data = response.json()
        first_journey = test_data["journeys"][0]
        walking_time = sum(
            leg['duration'] 
            for leg in first_journey['legs'] 
            if leg['mode']['id'] == 'walking')
        duration = first_journey["duration"]
        num_legs = len(first_journey['legs'])
        list_of_durations.append(duration)
        list_of_walking_time.append(walking_time)
    else:
        print(f"Failed for {postcode}: {response.status_code}")
        list_of_durations.append(None)
        list_of_walking_time.append(None)

In [83]:
print(len(list_of_walking_time) == len(list_of_durations) == len(most_deprived_list))
print(f"Durations: {list_of_durations}")
print(f"Walking times: {list_of_walking_time}")

True
Durations: [30, 65, 49, 71, 54, 66, 49, 63, 62, 71, 58, 53, 45, 49, 76, 91, 74, 59, 42, 53, 58, 56, 55, 68, 52, 61, 87, 33, 74, 43, 58, 42, 42]
Walking times: [20, 32, 23, 21, 26, 17, 33, 20, 17, 37, 23, 27, 28, 15, 17, 17, 20, 12, 21, 20, 22, 12, 18, 9, 19, 20, 24, 12, 24, 15, 17, 23, 17]


In [84]:
most_deprived["duration_to_central"] = list_of_durations
most_deprived["total_walking"] = list_of_walking_time
most_deprived.head()

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,lat,long,lep1,lep2,pfa,imd,calncv,stp,duration_to_central,total_walking
30863,E1 7AA,200012,NaN,E99999999,E99999999,E09000001,E05009308,E43000191,0,533623,...,51.515567,-0.075635,E37000051,NaN,E23000034,8678,E56000028,E54000029,30,20
96651,IG11 0AG,200608,NaN,E99999999,E99999999,E09000002,E05000039,E43000192,0,546206,...,51.531241,0.106421,E37000051,NaN,E23000001,2669,E56000028,E54000029,65,32
146587,NW11 9EH,198001,NaN,E99999999,E99999999,E09000003,E05000053,E43000193,0,524068,...,51.573205,-0.211092,E37000051,NaN,E23000001,2878,E56000027,E54000028,49,23
28219,DA8 2AB,198001,NaN,E99999999,E99999999,E09000004,E05011231,E43000194,0,551677,...,51.478252,0.182770,E37000051,NaN,E23000001,3591,E56000010,E54000030,71,21
141897,NW10 0AB,200711,NaN,E99999999,E99999999,E09000005,E05000100,E43000195,0,521198,...,51.553022,-0.253283,E37000051,NaN,E23000001,1192,E56000021,E54000027,54,26


In [85]:
most_deprived.to_csv("data/raw/most_deprived_postcodes.csv", index=False)

In [ ]:
# Now for Least Deprived
list_of_durations = []
list_of_walking_time = []
for postcode in least_deprived_list:
    base_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{postcode}/to/WC2A2AE"

    params = {
        "date": CURRENTDATE,
        "time": "0800",
        "app_id": API_KEY
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        test_data = response.json()
        first_journey = test_data["journeys"][0]
        walking_time = sum(
            leg['duration'] 
            for leg in first_journey['legs'] 
            if leg['mode']['id'] == 'walking')
        duration = first_journey["duration"]
        num_legs = len(first_journey['legs'])
        list_of_durations.append(duration)
        list_of_walking_time.append(walking_time)
    else:
        print(f"Failed for {postcode}: {response.status_code}")
        list_of_durations.append(None)
        list_of_walking_time.append(None)

In [87]:
print(len(list_of_walking_time) == len(list_of_durations) == len(least_deprived_list))
print(f"Durations: {list_of_durations}")
print(f"Walking times: {list_of_walking_time}")

True
Durations: [28, 55, 58, 65, 58, 56, 43, 54, 53, 66, 65, 44, 48, 51, 70, 67, 67, 51, 31, 37, 62, 52, 57, 52, 47, 59, 69, 39, 87, 49, 60, 52, 44]
Walking times: [10, 22, 23, 14, 27, 13, 25, 15, 17, 35, 28, 21, 23, 14, 30, 24, 13, 26, 22, 15, 11, 34, 20, 18, 21, 27, 24, 27, 22, 22, 16, 24, 24]


In [88]:
least_deprived["duration_to_central"] = list_of_durations
least_deprived["total_walking"] = list_of_walking_time
least_deprived.head()

,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,usertype,oseast1m,...,lat,long,lep1,lep2,pfa,imd,calncv,stp,duration_to_central,total_walking
61297,EC1Y 4AG,200902,NaN,E99999999,E99999999,E09000001,E05009299,E43000191,1,532601,...,51.520686,-0.090178,E37000051,NaN,E23000034,30379,E56000028,E54000029,28,10
97917,IG11 9AA,198001,NaN,E99999999,E99999999,E09000002,E05000035,E43000192,0,545624,...,51.541338,0.098494,E37000051,NaN,E23000001,17580,E56000028,E54000029,55,22
126980,N20 8DN,200106,NaN,E99999999,E99999999,E09000003,E05000059,E43000193,0,525780,...,51.630455,-0.184129,E37000051,NaN,E23000001,31544,E56000027,E54000028,58,23
25952,DA5 1DY,201404,NaN,E99999999,E99999999,E09000004,E05011229,E43000194,0,549340,...,51.440866,0.147384,E37000051,NaN,E23000001,32132,E56000010,E54000030,65,14
82219,HA3 0PF,198001,NaN,E99999999,E99999999,E09000005,E05000093,E43000195,0,517822,...,51.577084,-0.301107,E37000051,NaN,E23000001,25260,E56000021,E54000027,58,27


In [89]:
least_deprived.to_csv("data/raw/least_deprived_postcodes.csv", index=False)